In [ ]:
# 1) Configuration for non-instruction inference
import os
import logging

BASE_MODEL_ID = 'Qwen/Qwen2.5-0.5B'
ADAPTER_REPO_ID = 'Swapperss/finance-non-instruction-lora'
MAX_NEW_TOKENS = 160

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('non_instruction_inference')
logger.info('Notebook configuration loaded')
print('Base model:', BASE_MODEL_ID)
print('Adapter repo:', ADAPTER_REPO_ID)

In [ ]:
# 2) Optional Hugging Face login for private repos
from huggingface_hub import login

hf_token = os.getenv('HF_TOKEN', '').strip()
if hf_token:
    login(token=hf_token)
    print('Logged in using HF_TOKEN environment variable.')
else:
    print('HF_TOKEN not found. If repo is private, run:')
    print('from huggingface_hub import notebook_login; notebook_login()')

In [ ]:
# 3) Load tokenizer and model from Hugging Face Hub
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

use_cuda = torch.cuda.is_available()
print('CUDA available:', use_cuda)
if use_cuda:
    print('GPU:', torch.cuda.get_device_name(0))

try:
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO_ID, use_fast=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if use_cuda:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=quant_config,
        device_map='auto',
        trust_remote_code=True,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

model = PeftModel.from_pretrained(base_model, ADAPTER_REPO_ID)
model.eval()
logger.info('Loaded base model plus LoRA adapter from Hub')
print('Model loaded successfully.')

In [ ]:
# 4) Inference helper and sample runs
def generate_completion(prompt, max_new_tokens=MAX_NEW_TOKENS):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompts = [
    'Customers reported recurring service fee disputes in the last quarter,',
    'A consumer observed an unauthorized credit card transaction and',
    'When a payment is posted late due to bank processing delays,',
]

for idx, prompt in enumerate(prompts, start=1):
    print('=' * 90)
    print(f'PROMPT {idx}: {prompt}')
    print('OUTPUT:')
    print(generate_completion(prompt))